In [5]:
"""阶段一：比较异步 I/O 的串行执行和并发执行。"""

import asyncio
import time


async def fake_io(name: str, delay: float) -> str:
    """用 sleep 模拟一次外部 I/O 请求。"""
    await asyncio.sleep(delay)
    return f"{name}: done"

# 串行
async def run_serial() -> tuple[list[str], float]:
    started = time.perf_counter()
    results = [
        await fake_io("weather", 0.3),
        await fake_io("exchange_rate", 0.2),
        await fake_io("news", 0.1),
    ]
    return results, time.perf_counter() - started

# 并行
async def run_concurrent() -> tuple[list[str], float]:
    started = time.perf_counter()
    results = await asyncio.gather(
        fake_io("weather", 0.3),
        fake_io("exchange_rate", 0.2),
        fake_io("news", 0.1),
    )
    return results, time.perf_counter() - started


async def main() -> None:
    serial_results, serial_seconds = await run_serial()
    concurrent_results, concurrent_seconds = await run_concurrent()

    print(f"串行: {serial_seconds:.2f}s -> {serial_results}")
    print(f"并发: {concurrent_seconds:.2f}s -> {concurrent_results}")

    assert serial_results == concurrent_results
    assert serial_seconds >= 0.6
    assert concurrent_seconds < serial_seconds


await main()


串行: 0.60s -> ['weather: done', 'exchange_rate: done', 'news: done']
并发: 0.30s -> ['weather: done', 'exchange_rate: done', 'news: done']
